
1. Load and examine the data
2. Run with some models
3. Run with larger models and experiment with Jev
4. Create an agent with LangChain


In [42]:
import json
import os
import time
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import Markdown, display

import pandas as pd
import plotly.express as px
from pydantic import BaseModel, Field
from typing import Literal

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam
from openrouter import OpenRouter
from openrouter.components.decisionschoiceanswer import DecisionsChoiceAnswer

In [2]:
load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that always answers clearly and concisely."},
    {"role": "user", "content": "What is the best way to build skills for an AI FDE role?"}
]

response = openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

The best way to build skills for an **AI Forward Deployed Engineer (FDE)** role is to practice taking an ambiguous customer problem all the way to a working, reliable solution. The job sits at the intersection of software engineering, applied AI, and customer-facing problem solving.

### Build these skills in roughly this order

1. **Strong software fundamentals**
   - Be comfortable with Python and one production backend stack.
   - Build APIs, work with databases, write tests, use Git, and debug unfamiliar code.
   - Learn deployment basics: Docker, cloud services, logging, and monitoring.

2. **Applied AI engineering**
   - Use model APIs, but learn more than prompt writing: structured outputs, tool use, retrieval-augmented generation (RAG), and evaluations.
   - Understand failure modes such as hallucinations, latency, cost, and prompt injection.
   - Know when *not* to use an LLM, or when a simpler workflow is better.

3. **Data and integration work**
   - Practice connecting AI systems to real sources of truth: SQL databases, files, APIs, and business tools.
   - Learn data cleaning, permissions, and how to make answers traceable to their sources.

4. **Customer discovery and communication**
   - Turn a vague request into clear requirements and success metrics.
   - Ask about the user’s workflow, edge cases, constraints, and what “working” means.
   - Explain tradeoffs plainly, show progress early, and handle changing requirements.

5. **Production judgment**
   - Measure quality with representative examples, not just demos.
   - Add human review where appropriate, and design graceful fallbacks.
   - Track reliability, security, latency, and cost after deployment.

### A practical way to learn

Build **two or three end-to-end projects** around realistic workflows—for example, a support-ticket assistant, a document search tool, or an internal operations copilot. For each project:

- Start with a specific user problem and measurable goal.
- Build a thin working version quickly.
- Test it against a small evaluation set, including failure cases.
- Add integrations, permissions, logging, and a fallback path.
- Write a short case study explaining what you changed after testing.

A polished demo is useful, but a clear account of **how you found the real problem, evaluated the system, and made it dependable** is more compelling.

Finally, practice live problem-solving: take an unfamiliar workflow, ask clarifying questions, sketch a solution, and build a small prototype. That combination of technical execution and customer judgment is the core FDE skill.

In [4]:
from pathlib import Path
print(Path.cwd())

c:\Users\cloud\Documents\projects\ai-fde-agentic-architect\src\labs


In [5]:
def find_repo_root(start: Path = None) -> Path:
    """Walk up from start (default: cwd) until a folder containing pyproject.toml is found."""
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above " + str(start))

DATA = find_repo_root() / "data"

In [6]:



def load(name):
    """Read one JSON seed file."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))

def get_sample_data(tiny=False, held_out=False):
    """
    Return a list of claims, and the handler's decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the give claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")

    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


In [7]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [8]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

In [9]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

In [10]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"]
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


In [11]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


In [12]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))


**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


In [13]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

In [14]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

In [18]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's return policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached.  Work out the payout using secion 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the document disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""

def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record
    
{json.dumps(policy, indent=2)}

# Claim
{json.dumps(claim, indent=2)}
    """

def make_messages(claim) -> list[ChatCompletionMessageParam]:
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [24]:
def decide(claim, model, effort):
    """Ask one model to decide on claim."""
    start = time.time()
    # parse to call with structured outputs
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        # This ensures openrouter picks a model that can work with structured outputs
        extra_body={"provider": {"require_parameters": True}}
    )

    answer = response.choices[0].message.parsed
    if answer is None:
        raise ValueError(f"No parsed output for claim {claim['claim_id']}")
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": getattr(response.usage, "cost", None),
        "seconds": round(time.time() - start, 1),
    }


In [25]:
first = decide(claims[0], "openai/gpt-6-luna", "low")
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'Approve under Section 3: the carrier canceled the flight because of a crew problem, and the documented delay from the scheduled June 18 departure until the June 19 departure exceeds the Premium plan’s 6-hour waiting period. The attached carrier notice and receipts support the $420 in reasonable accommodation, meal and transport costs; this is within the $800 Trip Delay maximum, and no deductible applies.',
 'cost': 0.0003953,
 'seconds': 1.6}

In [26]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout when the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the label."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [27]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

openai/gpt-6-luna at low: 17 of 25 right for $0.0048 in 46 seconds


In [28]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

openai/gpt-6-luna at medium: 17 of 25 right for $0.0063 in 79 seconds


In [32]:
luna_medium = run("openai/gpt-6-luna", "high", claims, labels)

openai/gpt-6-luna at high: 16 of 25 right for $0.0072 in 118 seconds


In [29]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

google/gemini-3.5-flash-lite at low: 14 of 25 right for $0.0324 in 30 seconds


In [31]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

difficulty                           routine  hard  manual
model                        effort                       
openai/gpt-6-luna            low          13     2       2
                             medium       13     4       0
google/gemini-3.5-flash-lite low          14     0       0

In [33]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))

show_misses(luna_low, labels)


**CLM-1002, routine**: refer $0.00, where the handler said approve $580.00

- Model: The carrier’s report supports loss by a carrier during the covered trip, and the settlement letter confirms $400 paid. However, the claim includes shirts and a toiletry kit without receipts, and section 11 requires the documents named for the benefit; the submitted documents do not substantiate those items, so the claim must wait for the missing documentation. Section 5 applies the Premium $300 per-item limit, and section 10 requires deducting the carrier payment and the $50 deductible, but the payout cannot be finalized until the missing items are clarified.
- Handler: The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-item limit on Premium, so it counts at $300. The carrier's $400 comes off, then the $50 deductible.
- Tests: Payout arithmetic: the per-item limit, the carrier's payment and the deductible in order


**CLM-1004, routine**: refer $0.00, where the handler said approve $186.00

- Model: Section 4 covers essential clothing and toiletries bought while waiting for checked baggage delivered at least 12 hours after arrival; the documented delay was over 31 hours, and the carrier report and receipts are attached. However, section 4 excludes this benefit on the journey home, and the claim does not establish whether this arrival was an outbound journey or the journey home, so the wording and evidence do not settle eligibility.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1011, hard**: approve $1,040.00, where the handler said approve $1,150.00

- Model: Approve under section 5: the carrier report and settlement letter establish that the checked bag was lost by the carrier, and the attached receipts and card statement support the claimed belongings. Applying section 10, eligible amounts are $1,440 after per-item limits (including the suit and camera-with-lens as sets); subtracting the carrier’s $350 payment and the Premium $50 deductible gives $1,040, below the $1,500 baggage maximum.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The nine-hour delay exceeded the Premium plan’s six-hour waiting period, and the carrier confirms it was caused by a crew problem, a covered cause under Section 3. The required carrier confirmation and receipts are attached; the $210 in claimed reasonable delay expenses is below the $800 benefit maximum, with no deductible under Section 10.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The loss of the sunglasses may be covered under section 5, but the claim describes them being taken and requires a police report: theft must be reported to local police within 24 hours, with a copy of the report attached. The customer says police would not take a report and has provided a guide’s letter instead, so the required document is missing and the claim should be referred for review.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip and was reported to the police the same day, with a police report attached, as required by section 5. The eligible item amounts total $1,103; Electronics Plus raises the per-item limit for the electronics, and after the $50 Premium baggage deductible under sections 1 and 10, the payout is $1,053, below the $1,500 benefit maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: approve $85.00, where the handler said approve $155.00

- Model: The carrier confirms a severe-weather delay exceeding the First plan’s 3-hour waiting period, so the delay qualifies under section 3. The $85 rebooking fee is supported by a receipt and is an eligible reasonable rebooking cost; the $140 meal claim has no receipts as section 3 requires, so it cannot be paid on the documents provided.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The delay was 9 hours 17 minutes and the carrier certificate confirms a mechanical fault, meeting the First plan’s 3-hour waiting period under section 3. The meal receipts and carrier confirmation are attached; the $61 in reasonable meal expenses is eligible, and no deductible applies to Trip Delay under section 1, so section 10 yields $61.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


In [34]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model                        effort                          
openai/gpt-6-luna            low        17  0.004806     46.3
                             medium     17  0.006341     79.0
google/gemini-3.5-flash-lite low        14  0.032413     29.7

In [35]:
sol = run("openai/gpt-6-sol", "high", claims, labels)

openai/gpt-6-sol at high: 20 of 25 right for $0.1271 in 106 seconds


In [38]:
astra = run("openai/gpt-6-astra", "high", claims, labels)

openai/gpt-6-astra at high: 19 of 25 right for $0.6377 in 197 seconds


### Jev

Jev, from TypeSafe AI, doesn't generate text. We send it the facts as `state` along with a typed question, and it returns a probability for each option and a confidence. It chooses between approve, decline and refer in a fraction of a second. It can't write a rationale or work out a payout, so we score it on the decision alone.

Jev is on OpenRouter's Decisions endpoint, which the OpenAI client can't reach. We use OpenRouter's own Python client with the same key.

In [36]:
JEV = "typesafe/jev-1.13"

JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance.  Decide 'claim' using 'policy_wording' and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, becasue a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)

In [41]:
state = {"policy_wording": policy_wording, "policy": policies[claims[0]["policy_id"]], "claim": claims[0]}
response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
answer = response.answers["decision"]
print(type(answer).__module__, type(answer).__name__)

openrouter.components.decisionschoiceanswer DecisionsChoiceAnswer


In [43]:
def decide_with_jev(claim):
    """Ask Jev for the decision one on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    if not isinstance(answer, DecisionsChoiceAnswer): 
        raise ValueError(f"No decision returned for claim {claim['claim_id']}")
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[claim["claim_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2)
    }

In [46]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result["decision"] == labels[result["claim_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Jev: 17 of 25 decisions right for $0.0033 in 6 seconds
Luna at low: 20 of 25 decisions right


In [47]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

,sum,count
confident,,
False,13,20
True,4,5


In [54]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in a clear and concise manner."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

There’s no single, objectively verified answer; the "meaning" of the universe depends on the lens you look through:

**Science** describes the universe in terms of physics, chemistry, and cosmology: it arose from the Big Bang, evolves according to natural laws, and may eventually face heat death. It doesn’t assign purpose, but it reveals a process of increasing complexity, emergence, and interconnectedness.

**Philosophy** ranges widely:
- *Existentialism* holds that the universe has no built-in meaning, so individuals must create their own through choice, action, and values.
- *Nihilism* suggests there’s no intrinsic meaning or purpose at all.
- *Teleological* views argue the cosmos has an inherent direction or goal (e.g., evolution toward consciousness, cosmic progression).

**Religion & spirituality** typically frame the universe as created or sustained by a divine principle, with meaning tied to spiritual growth, karmic law, service, or alignment with a cosmic order.

**Human experience** often finds meaning in what we can observe, understand, and connect with: curiosity, art, relationships, the pursuit of knowledge, and the simple fact that conscious beings can ask the question at all. From this angle, the universe becomes a stage for meaning-making rather than a pre-scripted narrative.

In short: the universe may not come with a meaning pre-installed, but it provides the conditions—consciousness, time, and scale—for us to define, seek, or create one.

In [55]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [56]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

In [57]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_SErKrK43xlqvXGmGG6u7RoKj)
 Call ID: call_SErKrK43xlqvXGmGG6u7RoKj
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_5gLwJAMEB82SPLFaPtXqqKfE)
 Call ID: call_5gLwJAMEB82SPLFaPtXqqKfE
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: The carrier-confirmed cancellation was caused by a c

In [58]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

25 of 25 claims have a recorded decision


In [59]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM           17  0.004806     46.3
                         Agent         17  0.008668    148.6

In [60]:
show_misses(agent_results, labels)

**CLM-1004, routine**: approve $74.00, where the handler said approve $186.00

- Model: The carrier report confirms the checked bag was delivered 31 hours 15 minutes after arrival, exceeding the Premium plan’s 12-hour Baggage Delay waiting period, and receipts are attached. Under section 4, essential clothing and toiletries are covered; the underwear and basic toiletries ($27 + $47) qualify, but the dress bought for a welcome dinner is not shown to be essential, so it is excluded. The $74 payout is within the $300 limit and no deductible applies.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The theft occurred during the covered trip, and the attached Carabinieri report confirms the theft and was made within 24 hours as required by section 5. Under section 10, the Premium per-item limit reduces the $420 jacket to $300; the $80 daypack remains eligible, and subtracting the $50 baggage deductible gives a payout of $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The carrier confirmation documents a 9-hour-20-minute delay caused by a crew duty-time limit, meeting the Premium plan’s 6-hour waiting period and the covered crew-problem cause under section 3. Receipts and carrier confirmation are attached; the $210 in claimed accommodation, meal and transport costs is within the $800 Trip Delay maximum, and no deductible applies.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: Section 5 requires theft to be reported to local police within 24 hours and a copy of the police report; the submitted documents include only a tour-guide letter, not a police report. Please refer for review because the claimant says police would not take a report, and the wording does not specify whether an alternative document can satisfy this requirement.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip and was reported to police the same day, with the required police report attached; the report matches the account and identifies the listed contents. The eligible item amounts total $1,103, within the Premium baggage maximum, and Electronics Plus raises the electronics per-item limit; after the $50 baggage deductible under sections 1, 5, 8 and 10, the payout is $1,053.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The carrier report confirms a severe-weather delay exceeding the First plan’s 3-hour waiting period, and the $85 rebooking receipt is attached. Section 3 requires receipts for extra costs, but receipts are missing for the $140 meal claim; refer for the required documentation before determining the payable amount.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and the same-day police report satisfies section 5’s theft reporting requirement; the claim was also filed within 90 days. Under sections 1 and 10, the Premium per-item limit reduces the eligible $899 claim to $300, then the $50 baggage deductible applies, for a payout of $250.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: The carrier certificate confirms a 14-hour-15-minute delay caused by a crew problem, exceeding the Economy plan’s 12-hour waiting period under section 3. The meal receipts are attached and total $71; the return journey is covered beyond the return date under section 3, and no deductible applies to Trip Delay.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


In [62]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, sol, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,speed
0,openai/gpt-6-luna,low,LLM,17 / 25,$0.0048,1.9 seconds a claim
1,openai/gpt-6-luna,high,LLM,16 / 25,$0.0072,4.7 seconds a claim
2,google/gemini-3.5-flash-lite,low,LLM,14 / 25,$0.0324,1.2 seconds a claim
3,openai/gpt-6-sol,high,LLM,20 / 25,$0.1271,4.2 seconds a claim
4,openai/gpt-6-luna,low,Agent,17 / 25,$0.0087,5.9 seconds a claim
